# PadelVision — 05 Ball tracking (stage 4, Kaggle)

Runs the trained `ball-detector` (GitHub release `ball-yolo11s-v1`) on **every frame**, then tracks the
**ball in play**: detections are linked into trajectories; static ones (spare balls on the floor, fixed false spots)
and ones outside the court region are dropped; gaps of up to 5 frames are filled and marked `interpolated`.
Positions are image pixels (an airborne ball has no court position).

Outputs: `ball.parquet`, `ball_stats.json`, a check against the reviewed eval labels, a 30 s preview with the ball
drawn, and `ball_tracking.zip` to download. Ball tracking can then be re-tuned locally with
`padelvision ball-retrack` without the GPU.

**Setup (Kaggle):** GPU T4 (x2 not needed), Internet on, video attached. **Run All** takes about 5–10 min.

In [ ]:
import os, subprocess, sys
ON_KAGGLE = os.path.exists("/kaggle")
ON_COLAB = "google.colab" in sys.modules
WORK = "/kaggle/working" if ON_KAGGLE else "/content"
print(f"Kaggle={ON_KAGGLE} Colab={ON_COLAB} workdir={WORK}")
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU: enable one in the notebook settings"

In [ ]:
REPO_URL = "https://github.com/Duggineniakhil/PadelVision.git"
BRANCH = "v2-rebuild"
REPO = f"{WORK}/PadelVision"

if os.path.exists(REPO):
    !git -C {REPO} pull --ff-only
else:
    !git clone --depth 1 -b {BRANCH} {REPO_URL} {REPO}
!pip install -q -e "{REPO}/pipeline[ml]"

# Editable installs aren't picked up by an already-running kernel; make the package importable now.
sys.path.insert(0, f"{REPO}/pipeline/src")
import padelvision
print("padelvision", padelvision.__version__)

In [ ]:
!cd {REPO} && python scripts/fetch_models.py

In [ ]:
from pathlib import Path

if ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

# Leave empty to use the first video found under /kaggle/input (Kaggle) or MyDrive/padel (Colab).
VIDEO_PATH = ""

if not VIDEO_PATH:
    search_root = Path("/kaggle/input") if ON_KAGGLE else Path("/content/drive/MyDrive/padel")
    videos = sorted(p for ext in ("*.mp4", "*.mov", "*.avi", "*.mkv") for p in search_root.rglob(ext))
    print("Videos found:", *videos, sep="\n  ")
    assert videos, f"No video found under {search_root}; attach one or set VIDEO_PATH"
    VIDEO_PATH = str(videos[0])

assert os.path.exists(VIDEO_PATH), f"Not found: {VIDEO_PATH}"
print("Using video:", VIDEO_PATH)

In [ ]:
# court.json: pasted below, else ml/calibrations/<video name>.json in the repo, else court.json next to the video.
COURT_JSON_TEXT = """
"""

court_files = sorted(Path("/kaggle/input" if ON_KAGGLE else "/content/drive/MyDrive/padel").rglob("court.json"))
# Calibrations committed to the repo, named after the video file (e.g. ml/calibrations/Test_video.json).
repo_cal = Path(REPO) / "ml" / "calibrations" / f"{Path(VIDEO_PATH).stem}.json"
if repo_cal.exists():
    court_files.insert(0, repo_cal)
COURT_JSON = f"{WORK}/court.json"
if COURT_JSON_TEXT.strip():
    Path(COURT_JSON).write_text(COURT_JSON_TEXT)
elif court_files:
    Path(COURT_JSON).write_text(court_files[0].read_text())
    print("Using", court_files[0])
else:
    raise FileNotFoundError("No court.json: attach it to the dataset or paste it above")

from padelvision.court import CourtCalibration
cal = CourtCalibration.load(COURT_JSON)
print(f"court error {cal.reprojection_error_m()*100:.0f} cm, lens k1={cal.lens.k1:.3f} k2={cal.lens.k2:.3f}")

## 1. Detect + track

In [ ]:
RUN_DIR = f"{WORK}/runs/match1"
MAX_FRAMES = None   # None = whole video
WITH_PLAYERS = True # also run player analytics (~3 min) so the preview shows players + ball
Path(RUN_DIR).mkdir(parents=True, exist_ok=True)

extra = f"--max-frames {MAX_FRAMES}" if MAX_FRAMES else ""
if WITH_PLAYERS and not Path(f"{RUN_DIR}/players.parquet").exists():
    !cd {REPO} && python -m padelvision.cli analyze "{VIDEO_PATH}" --court {COURT_JSON} --out {RUN_DIR} {extra}
else:
    import shutil
    shutil.copy(COURT_JSON, f"{RUN_DIR}/court.json")  # the ball tracker uses it for the court region
!cd {REPO} && python -m padelvision.cli ball-track "{VIDEO_PATH}" --run {RUN_DIR} {extra}

## 2. Summary

In [ ]:
import json
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from padelvision.ball.evaluate import classify, review_crops

ball = pd.read_parquet(f"{RUN_DIR}/ball.parquet")
stats = json.loads(Path(f"{RUN_DIR}/ball_stats.json").read_text())
print(json.dumps(stats, indent=1))

fps = cv2.VideoCapture(VIDEO_PATH).get(cv2.CAP_PROP_FPS)
per_s = ball.groupby((ball.frame / fps).astype(int)).state.value_counts().unstack(fill_value=0)
per_s.plot(kind="bar", stacked=True, width=1.0, figsize=(15, 2.8), color={"detected": "#2b8cbe", "interpolated": "#fdae61"})
plt.xticks([]); plt.xlabel("second"); plt.ylabel("frames"); plt.title("Ball in play per second (max = fps)")
plt.show()

## 3. Check against reviewed labels

In [ ]:
# Check against the pooled, visually reviewed labels (only frames that were reviewed count).
LABELS = pd.read_csv(f"{REPO}/ml/eval/labels/Test_video_ball.csv")
EVAL = json.loads(Path(f"{REPO}/ml/eval/labels/Test_video_ball_frames.json").read_text())
EVAL_FRAMES = [f for f in EVAL["frames"] if f not in set(EVAL["excluded"])]
on_eval = ball[ball.frame.isin(EVAL_FRAMES)].assign(conf=lambda d: d.conf.fillna(0.0))
judged = classify(on_eval[["frame", "u", "v", "conf"]], LABELS)
judged["state"] = on_eval.state.values
summary = judged.groupby(["state", "outcome"]).size().unstack(fill_value=0)
display(summary)
tp, fp = (judged.outcome == "tp").sum(), (judged.outcome == "fp").sum()
print(f"eval frames with a tracked ball: {len(on_eval)} of {len(EVAL_FRAMES)}; "
      f"precision on reviewed spots: {tp / max(1, tp + fp):.2f} ({tp} ball, {fp} not ball); "
      f"unverified: {(judged.outcome == 'unverified').sum()}")

need = set(on_eval.frame)
FRAMES, cap, f = {}, cv2.VideoCapture(VIDEO_PATH), 0
while need and f <= max(need):
    ok, img = cap.read()
    if not ok:
        break
    if f in need:
        FRAMES[f] = img
    f += 1
cap.release()
OUT = Path(f"{WORK}/ball_tracking")
(OUT / "crops").mkdir(parents=True, exist_ok=True)
review_crops(on_eval[["frame", "u", "v", "conf"]], LABELS, FRAMES, OUT / "crops", "tracked", max_crops=120)
judged.to_csv(OUT / "eval_judged.csv", index=False)

## 4. Preview (players + ball)

In [ ]:
PREVIEW_START_S = 0
!cd {REPO} && python -m padelvision.cli render "{VIDEO_PATH}" --run {RUN_DIR} --start {PREVIEW_START_S} --seconds 30 --out {RUN_DIR}/preview_raw.mp4
!ffmpeg -y -loglevel error -i {RUN_DIR}/preview_raw.mp4 -vcodec libx264 -pix_fmt yuv420p -crf 28 {RUN_DIR}/preview.mp4 && rm {RUN_DIR}/preview_raw.mp4

# Contact sheet of 12 preview frames (so the result can be checked without playing the video).
cap = cv2.VideoCapture(f"{RUN_DIR}/preview.mp4")
n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
tiles = []
for k in np.linspace(0, n - 1, 12).astype(int):
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(k))
    ok, img = cap.read()
    if ok:
        tiles.append(cv2.resize(img, (640, 360)))
cap.release()
sheet = np.vstack([np.hstack(tiles[i:i + 3]) for i in range(0, len(tiles) - len(tiles) % 3, 3)])
cv2.imwrite(str(OUT / "preview_sheet.jpg"), sheet, [cv2.IMWRITE_JPEG_QUALITY, 85])

from IPython.display import Video
Video(f"{RUN_DIR}/preview.mp4", embed=True, width=960)

## 5. Download

In [ ]:
import shutil
from IPython.display import FileLink
for name in ["ball_detections.parquet", "ball_detections.json", "ball.parquet", "ball_stats.json",
             "video.json", "court.json", "players.parquet", "stats.json"]:
    p = Path(RUN_DIR) / name
    if p.exists():
        shutil.copy(p, OUT / name)
ZIP = f"{WORK}/ball_tracking.zip"
shutil.make_archive(ZIP[:-4], "zip", OUT.parent, OUT.name)
print(f"{ZIP}: {Path(ZIP).stat().st_size / 1e6:.1f} MB")
FileLink(Path(ZIP).relative_to(Path.cwd()) if Path(ZIP).is_relative_to(Path.cwd()) else ZIP)